### Creating the Medallion Architecture

This notebook builds the **Bronze layer** for the aviation dataset. Source tables are read as-is from the `dbacademy` catalog and written into the `aviation_bronze` schema, with no transformations applied at this stage.

#### Load Source Tables
Read the airline, airport, flight, and delay tables from the `dbacademy.default` catalog.

In [0]:
airline_df = spark.table("dbacademy.default.airline_dim")
airport_df = spark.table("dbacademy.default.airport_dim")
flight_df = spark.table("dbacademy.default.flight_fact")
delay_df = spark.table("dbacademy.default.delay_fact")

#### Row Counts
Sanity-check row counts for each table before going further.

In [0]:
print("Airline :", airline_df.count())
print("Airport :", airport_df.count())
print("Flight :", flight_df.count())
print("Delay :", delay_df.count())

Airline : 18
Airport : 380
Flight : 2000000
Delay : 2000000


#### Schema Check

In [0]:
airline_df.printSchema()
airport_df.printSchema()
flight_df.printSchema()
delay_df.printSchema()

root
 |-- AIRLINE: string (nullable = true)
 |-- AIRLINE_DOT: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- DOT_CODE: double (nullable = true)

root
 |-- AIRPORT_CODE: string (nullable = true)
 |-- CITY: string (nullable = true)

root
 |-- flight_id: long (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- FL_NUMBER: double (nullable = true)
 |-- ORIGIN_AIRPORT_CODE: string (nullable = true)
 |-- DEST_AIRPORT_CODE: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)

root
 |-- flight_id: long (nullable = true)
 |-- DELAY_DUE_CARRIER: double (nullable = true)
 |-- DELA

#### Preview Sample Records

In [0]:
display(airline_df.limit(10))
display(airport_df.limit(10))
display(flight_df.limit(10))
display(delay_df.limit(10))

AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE
Envoy Air,Envoy Air: MQ,MQ,20398.0
Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790.0
JetBlue Airways,JetBlue Airways: B6,B6,20409.0
Mesa Airlines Inc.,Mesa Airlines Inc.: YV,YV,20378.0
Hawaiian Airlines Inc.,Hawaiian Airlines Inc.: HA,HA,19690.0
Frontier Airlines Inc.,Frontier Airlines Inc.: F9,F9,20436.0
PSA Airlines Inc.,PSA Airlines Inc.: OH,OH,20397.0
American Airlines Inc.,American Airlines Inc.: AA,AA,19805.0
Horizon Air,Horizon Air: QX,QX,19687.0
United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977.0


AIRPORT_CODE,CITY
PGD,"PUNTA GORDA, FL"
DAB,"DAYTONA BEACH, FL"
CWA,"MOSINEE, WI"
ESC,"ESCANABA, MI"
LAR,"LARAMIE, WY"
BZN,"BOZEMAN, MT"
TUS,"TUCSON, AZ"
JNU,"JUNEAU, AK"
CHA,"CHATTANOOGA, TN"
DCA,"WASHINGTON, DC"


flight_id,FL_DATE,AIRLINE_CODE,FL_NUMBER,ORIGIN_AIRPORT_CODE,DEST_AIRPORT_CODE,CRS_DEP_TIME,DEP_TIME,CRS_ARR_TIME,ARR_TIME,DEP_DELAY,ARR_DELAY,DISTANCE,CANCELLED,DIVERTED
34359738368,2021-09-27,UA,654.0,FLL,IAD,1845.0,1837.0,2118.0,2103.0,-8.0,-15.0,901.0,0.0,0.0
34359738369,2021-06-10,G4,193.0,LAS,TUS,620.0,611.0,739.0,730.0,-9.0,-9.0,365.0,0.0,0.0
34359738370,2019-11-07,B6,1453.0,FLL,SJU,1645.0,1649.0,2017.0,2004.0,4.0,-13.0,1046.0,0.0,0.0
34359738371,2023-04-20,OO,3892.0,SEA,BOI,1158.0,1157.0,1430.0,1428.0,-1.0,-2.0,399.0,0.0,0.0
34359738372,2019-08-06,B6,660.0,PHL,BOS,1608.0,1657.0,1738.0,1820.0,49.0,42.0,280.0,0.0,0.0
34359738373,2019-07-12,F9,1659.0,PWM,RDU,1713.0,1652.0,1922.0,1851.0,-21.0,-31.0,700.0,0.0,0.0
34359738374,2022-12-08,YX,4696.0,LGA,CHO,759.0,750.0,932.0,928.0,-9.0,-4.0,305.0,0.0,0.0
34359738375,2019-05-24,HA,102.0,HNL,ITO,500.0,456.0,550.0,542.0,-4.0,-8.0,216.0,0.0,0.0
34359738376,2022-10-22,WN,717.0,RNO,LGB,1115.0,1116.0,1235.0,1239.0,1.0,4.0,402.0,0.0,0.0
34359738377,2023-02-24,AA,1194.0,DFW,ILM,1246.0,1253.0,1618.0,1637.0,7.0,19.0,1106.0,0.0,0.0


flight_id,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,null,null,null,null,null
1,null,null,null,null,null
2,null,null,null,null,null
3,53.0,0.0,0.0,0.0,0.0
4,63.0,0.0,97.0,0.0,0.0
5,null,null,null,null,null
6,null,null,null,null,null
7,null,null,null,null,null
8,null,null,null,null,null
9,null,null,null,null,null


#### Create the Bronze Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aviation_bronze;

#### Drop Existing Tables
Ensures a clean run by clearing out anything left over from a previous execution.

In [0]:
%sql
DROP TABLE IF EXISTS airline_dim_raw;
DROP TABLE IF EXISTS airport_dim_raw;
DROP TABLE IF EXISTS delay_fact_raw;
DROP TABLE IF EXISTS flight_fact_raw;

#### Confirm the Schema is Empty

In [0]:
%sql
use aviation_bronze;
show tables;

database,tableName,isTemporary
aviation_bronze,airline_dim_raw,false
aviation_bronze,airport_dim_raw,false
aviation_bronze,delay_fact_raw,false
aviation_bronze,flight_fact_raw,false


#### Write Tables to the Bronze Layer
Persist each dataframe as a managed table in `aviation_bronze`, overwriting on every run.

In [0]:
airline_df.write \
    .mode("overwrite") \
    .saveAsTable("aviation_bronze.airline_dim_raw")

airport_df.write \
    .mode("overwrite") \
    .saveAsTable("aviation_bronze.airport_dim_raw")

flight_df.write \
    .mode("overwrite") \
    .saveAsTable("aviation_bronze.flight_fact_raw")

delay_df.write \
    .mode("overwrite") \
    .saveAsTable("aviation_bronze.delay_fact_raw")

#### Verify Row Counts After Write

In [0]:
spark.table("aviation_bronze.airline_dim_raw").count()

18

In [0]:
spark.table("aviation_bronze.airport_dim_raw").count()

380

In [0]:
spark.table("aviation_bronze.flight_fact_raw").count()

2000000

In [0]:
spark.table("aviation_bronze.delay_fact_raw").count()

2000000

### Bronze Layer Completed

**Tables created:**
1. `airline_dim_raw`
2. `airport_dim_raw`
3. `flight_fact_raw`
4. `delay_fact_raw`

**Source format:** Parquet

**Purpose:** Store raw, curated data before quality validation and cleansing.